# Stage-Wise Curvature Tuning on Beans

The original Curvature Tuning paper was first reproduced separately on the Beans dataset using the Baseline and Single-Parameter Curvature Tuning (S-CT) methods.

The three-seed reproduction produced:

- Baseline: **89.32 ± 1.61%**
- S-CT: **91.15 ± 0.37%**
- Selected S-CT β values: **0.78, 0.77, 0.79**
- Mean selected β: **0.78**

Those reproduction experiments are preserved in the original reproduction branch/results.

This branch focuses on the proposed extension: **Stage-Wise Curvature Tuning (SW-CT)**.

Instead of using one global β value across the network, SW-CT introduces four trainable stage-wise β parameters:

- β₁: stem + layer1
- β₂: layer2
- β₃: layer3
- β₄: layer4

The pretrained backbone remains frozen, the classifier is trained for the downstream task, and the four β parameters are learned through backpropagation. The CTU coefficient \(c\) is fixed at 0.5.

The initial SW-CT experiment starts all four β parameters at **0.80** and evaluates seeds 42, 43, and 44.

In [ ]:
!nvidia-smi

Mon Sep 28 19:59:58 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   40C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
%cd /content
!git clone https://github.com/Syed1611/curvature-tuning-research.git
%cd /content/curvature-tuning-research/src/curvature-tuning
!git branch --show-current

/content
fatal: destination path 'curvature-tuning-research' already exists and is not an empty directory.


/content/curvature-tuning-research/src/curvature-tuning
stage-wise-ct


## Environment Setup

Some package dependency warnings may appear during installation because the Colab environment contains additional preinstalled packages. The versions required for this research are installed explicitly below.

After installation, NumPy is forced to version 1.26.3 for compatibility with the experiment environment.

In [ ]:
%pip install -q -r requirements.txt

%pip install --force-reinstall --no-cache-dir "numpy==1.26.3"

  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.4/42.4 kB 5.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.6/57.6 kB 7.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.9/89.9 kB 13.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 133.2/133.2 kB 18.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 85.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.4/78.4 kB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.1/13.1 MB 86.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.3/8.3 MB 87.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.7/9.7 MB 19.4 MB/s eta 0:00:0000:0100:01m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.3/13.3 MB 122.6 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.5/62.5 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 487.

### Runtime Restart Required

After installing the dependencies and NumPy 1.26.3, restart the Colab runtime so that the newly installed package versions are loaded correctly.

**After restarting, do not rerun the installation cells above. Start from the next cell, which returns to the repository directory and configures the environment.**

In [ ]:
%cd /content/curvature-tuning-research/src/curvature-tuning

import os
os.environ["WANDB_MODE"] = "disabled"

print("Working directory:", os.getcwd())

/content/curvature-tuning-research/src/curvature-tuning
Working directory: /content/curvature-tuning-research/src/curvature-tuning


In [ ]:
import torch
import datasets
import numpy
import pandas
import sklearn
import tqdm
import loguru

print("Torch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))
print("datasets:", datasets.__version__)
print("NumPy:", numpy.__version__)
print("Pandas:", pandas.__version__)
print("sklearn:", sklearn.__version__)
print("tqdm:", tqdm.__version__)
print("loguru:", loguru.__version__)

Torch: 2.6.0+cu124
CUDA: True
GPU: Tesla T4
datasets: 3.4.1
NumPy: 1.26.3
Pandas: 2.2.3
sklearn: 1.5.2
tqdm: 4.66.5
loguru: 0.7.2


In [ ]:
from utils.data import get_data_loaders

train_loader, test_loader, val_loader = get_data_loaders(
    "imagenet_to_beans",
    train_batch_size=32,
    test_batch_size=800,
    seed=42
)

print("Train samples:", len(train_loader.dataset))
print("Validation samples:", len(val_loader.dataset))
print("Test samples:", len(test_loader.dataset))

images, labels = next(iter(train_loader))

print("Image batch shape:", images.shape)
print("Label batch shape:", labels.shape)
print("Labels:", labels[:10])

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


data/train-00000-of-00001.parquet:   0%|          | 0.00/144M [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/18.5M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/17.7M [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

Train samples: 1034
Validation samples: 133
Test samples: 128
Image batch shape: torch.Size([32, 3, 224, 224])
Label batch shape: torch.Size([32])
Labels: tensor([1, 1, 0, 1, 0, 2, 1, 0, 1, 1])


# From Original Curvature Tuning to Stage-Wise Curvature Tuning

Before developing the proposed method, the original Baseline and Single-Parameter Curvature Tuning (S-CT) experiments were reproduced on the Beans dataset using seeds 42, 43, and 44.

### Original reproduction results

- Baseline: **89.32 ± 1.61%**
- S-CT: **91.15 ± 0.37%**
- Best S-CT β values: **0.78, 0.77, 0.79**
- Mean selected β: **0.78**

These results show that using a single global curvature parameter can improve transfer-learning performance over the unchanged ReLU baseline.

## Proposed Method: 4-Parameter Stage-Wise Curvature Tuning

The original S-CT method uses one shared β value across all ReLU activations in the network. Our proposed modification allows different sections of ResNet-18 to learn different curvature values.

Four trainable β parameters are introduced:

- β₁: stem + layer1
- β₂: layer2
- β₃: layer3
- β₄: layer4

The pretrained backbone weights remain frozen. The linear classifier and four stage-wise β parameters are optimized during transfer learning, while the CTU coefficient \(c\) remains fixed at 0.5.

For the initial experiment, all stage-wise β values are initialized to **0.80**. The default β learning rate in the implementation is the **default** one.

The experiment is repeated using seeds 42, 43, and 44 to measure both performance and the consistency of the learned stage-wise curvature pattern.

In [ ]:
!python generalization_stagewise_ct.py \
    --model resnet18 \
    --pretrained_ds imagenet \
    --transfer_ds beans \
    --seed 42 \
    --init_beta 0.8

2026-09-23 22:48:16.013 | INFO     | __main__:main:245 - Log file: ./logs/generalization_stagewise_ct_imagenet_to_beans_resnet18_seed42_initbeta0.8_betalr0.1.log
2026-09-23 22:48:16.074 | INFO     | __main__:main:261 - Running on cuda
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth
100% 44.7M/44.7M [00:00<00:00, 193MB/s]
Stage-Wise CT ReLU counts: [3, 2, 2, 2]
2026-09-23 22:48:18.392 | INFO     | __main__:main:312 - Initial stage betas: [0.800000011920929, 0.800000011920929, 0.800000011920929, 0.800000011920929]
2026-09-23 22:48:18.393 | INFO     | __main__:main:328 - Trainable curvature parameters: 4
2026-09-23 22:48:18.393 | INFO     | __main__:main:333 - Total trainable parameters: 1543
2026-09-23 22:48:20.773 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 1.034333, Accuracy: 46.88%
2026-09-23 22:48:29.184 | INFO     | train:test_epoch:90 - Epoch 1, Val Loss: 0.718755, Val Accuracy: 72.1

In [ ]:
!python generalization_stagewise_ct.py \
    --model resnet18 \
    --pretrained_ds imagenet \
    --transfer_ds beans \
    --seed 43 \
    --init_beta 0.8

2026-09-23 22:51:23.633 | INFO     | __main__:main:245 - Log file: ./logs/generalization_stagewise_ct_imagenet_to_beans_resnet18_seed43_initbeta0.8_betalr0.1.log
2026-09-23 22:51:23.693 | INFO     | __main__:main:261 - Running on cuda
Stage-Wise CT ReLU counts: [3, 2, 2, 2]
2026-09-23 22:51:25.694 | INFO     | __main__:main:312 - Initial stage betas: [0.800000011920929, 0.800000011920929, 0.800000011920929, 0.800000011920929]
2026-09-23 22:51:25.694 | INFO     | __main__:main:328 - Trainable curvature parameters: 4
2026-09-23 22:51:25.695 | INFO     | __main__:main:333 - Total trainable parameters: 1543
2026-09-23 22:51:26.862 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 1.319479, Accuracy: 28.12%
2026-09-23 22:51:34.571 | INFO     | train:test_epoch:90 - Epoch 1, Val Loss: 0.762525, Val Accuracy: 65.41%
2026-09-23 22:51:34.572 | INFO     | __main__:transfer:127 - Epoch 1: val_acc=65.41, betas=[0.7834, 0.9062, 0.6667, 0.7894]
2026-09-23 22:51:34.590 | INFO     | __main__:

In [ ]:
!python generalization_stagewise_ct.py \
    --model resnet18 \
    --pretrained_ds imagenet \
    --transfer_ds beans \
    --seed 44 \
    --init_beta 0.8

2026-09-23 22:54:31.754 | INFO     | __main__:main:245 - Log file: ./logs/generalization_stagewise_ct_imagenet_to_beans_resnet18_seed44_initbeta0.8_betalr0.1.log
2026-09-23 22:54:31.815 | INFO     | __main__:main:261 - Running on cuda
Stage-Wise CT ReLU counts: [3, 2, 2, 2]
2026-09-23 22:54:34.177 | INFO     | __main__:main:312 - Initial stage betas: [0.800000011920929, 0.800000011920929, 0.800000011920929, 0.800000011920929]
2026-09-23 22:54:34.177 | INFO     | __main__:main:328 - Trainable curvature parameters: 4
2026-09-23 22:54:34.178 | INFO     | __main__:main:333 - Total trainable parameters: 1543
2026-09-23 22:54:35.374 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 1.206441, Accuracy: 34.38%
2026-09-23 22:54:42.557 | INFO     | train:test_epoch:90 - Epoch 1, Val Loss: 0.720485, Val Accuracy: 63.91%
2026-09-23 22:54:42.558 | INFO     | __main__:transfer:127 - Epoch 1: val_acc=63.91, betas=[0.8294, 0.8877, 0.6955, 0.5649]
2026-09-23 22:54:42.573 | INFO     | __main__:

In [ ]:
import json
import numpy as np

seeds = [42, 43, 44]

accs = []
betas = []

for seed in seeds:
    path = (
        f"results/"
        f"stage_ct_imagenet_to_beans_resnet18_seed{seed}_initbeta0.8_betalr0.1.json"
    )

    with open(path) as f:
        result = json.load(f)

    accs.append(result["accuracy"])
    betas.append(result["stage_betas"])

print("Stage-Wise CT")
print("Accuracies:", accs)
print(f"Mean accuracy: {np.mean(accs):.2f}%")
print(f"Std: {np.std(accs):.2f}")

print("\nStage betas:")
for seed, b in zip(seeds, betas):
    print(seed, [round(x, 4) for x in b])

print(
    "\nMean stage betas:",
    np.round(np.mean(np.array(betas), axis=0), 4)
)

Stage-Wise CT
Accuracies: [89.84375, 90.625, 90.625]
Mean accuracy: 90.36%
Std: 0.37

Stage betas:
42 [0.8327, 0.9485, 0.9086, 0.6446]
43 [0.8478, 0.9478, 0.8341, 0.6126]
44 [0.8484, 0.9421, 0.8545, 0.6361]

Mean stage betas: [0.843  0.9461 0.8657 0.6311]


## β Learning-Rate Sensitivity: 0.03

The initial SW-CT experiment used the default β learning rate of 0.1. We now reduce the β learning rate to **0.03** while keeping the initialization fixed at β = 0.80.

This experiment tests whether slower curvature updates produce more stable or more accurate stage-wise solutions.

In [ ]:
!python generalization_stagewise_ct.py \
    --model resnet18 \
    --pretrained_ds imagenet \
    --transfer_ds beans \
    --seed 42 \
    --init_beta 0.8 \
    --beta_lr 0.03

2026-09-23 22:59:11.197 | INFO     | __main__:main:245 - Log file: ./logs/generalization_stagewise_ct_imagenet_to_beans_resnet18_seed42_initbeta0.8_betalr0.03.log
2026-09-23 22:59:11.298 | INFO     | __main__:main:261 - Running on cuda
Stage-Wise CT ReLU counts: [3, 2, 2, 2]
2026-09-23 22:59:13.280 | INFO     | __main__:main:312 - Initial stage betas: [0.800000011920929, 0.800000011920929, 0.800000011920929, 0.800000011920929]
2026-09-23 22:59:13.280 | INFO     | __main__:main:328 - Trainable curvature parameters: 4
2026-09-23 22:59:13.281 | INFO     | __main__:main:333 - Total trainable parameters: 1543
2026-09-23 22:59:14.534 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 1.034333, Accuracy: 46.88%
2026-09-23 22:59:21.673 | INFO     | train:test_epoch:90 - Epoch 1, Val Loss: 0.715784, Val Accuracy: 72.93%
2026-09-23 22:59:21.674 | INFO     | __main__:transfer:127 - Epoch 1: val_acc=72.93, betas=[0.8235, 0.7948, 0.8068, 0.7953]
2026-09-23 22:59:21.693 | INFO     | __main__

## β Learning-Rate Sensitivity: 0.01

The β learning rate of 0.03 did not produce a clear improvement, so the learning rate is reduced further to **0.01**.

All other settings remain unchanged, including the initial β = 0.80. Seed 42 is tested first before running the remaining seeds.

In [ ]:
!python generalization_stagewise_ct.py \
    --model resnet18 \
    --pretrained_ds imagenet \
    --transfer_ds beans \
    --seed 42 \
    --init_beta 0.8 \
    --beta_lr 0.01

2026-09-23 23:02:19.669 | INFO     | __main__:main:245 - Log file: ./logs/generalization_stagewise_ct_imagenet_to_beans_resnet18_seed42_initbeta0.8_betalr0.01.log
2026-09-23 23:02:19.732 | INFO     | __main__:main:261 - Running on cuda
Stage-Wise CT ReLU counts: [3, 2, 2, 2]
2026-09-23 23:02:22.052 | INFO     | __main__:main:312 - Initial stage betas: [0.800000011920929, 0.800000011920929, 0.800000011920929, 0.800000011920929]
2026-09-23 23:02:22.053 | INFO     | __main__:main:328 - Trainable curvature parameters: 4
2026-09-23 23:02:22.053 | INFO     | __main__:main:333 - Total trainable parameters: 1543
2026-09-23 23:02:24.160 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 1.034333, Accuracy: 46.88%
2026-09-23 23:02:31.486 | INFO     | train:test_epoch:90 - Epoch 1, Val Loss: 0.715264, Val Accuracy: 72.18%
2026-09-23 23:02:31.487 | INFO     | __main__:transfer:127 - Epoch 1: val_acc=72.18, betas=[0.8092, 0.7986, 0.8023, 0.7988]
2026-09-23 23:02:31.502 | INFO     | __main__

### Extending β Learning Rate 0.01 to All Seeds

Seed 42 showed a small improvement with β learning rate 0.01. To determine whether this behavior is consistent rather than seed-specific, the same configuration is now evaluated using seeds 43 and 44.

In [ ]:
!python generalization_stagewise_ct.py \
    --model resnet18 \
    --pretrained_ds imagenet \
    --transfer_ds beans \
    --seed 43 \
    --init_beta 0.8 \
    --beta_lr 0.01

2026-09-23 23:05:29.427 | INFO     | __main__:main:245 - Log file: ./logs/generalization_stagewise_ct_imagenet_to_beans_resnet18_seed43_initbeta0.8_betalr0.01.log
2026-09-23 23:05:29.500 | INFO     | __main__:main:261 - Running on cuda
Stage-Wise CT ReLU counts: [3, 2, 2, 2]
2026-09-23 23:05:31.466 | INFO     | __main__:main:312 - Initial stage betas: [0.800000011920929, 0.800000011920929, 0.800000011920929, 0.800000011920929]
2026-09-23 23:05:31.467 | INFO     | __main__:main:328 - Trainable curvature parameters: 4
2026-09-23 23:05:31.467 | INFO     | __main__:main:333 - Total trainable parameters: 1543
2026-09-23 23:05:32.710 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 1.319479, Accuracy: 28.12%
2026-09-23 23:05:41.288 | INFO     | train:test_epoch:90 - Epoch 1, Val Loss: 0.775484, Val Accuracy: 69.92%
2026-09-23 23:05:41.289 | INFO     | __main__:transfer:127 - Epoch 1: val_acc=69.92, betas=[0.8018, 0.8167, 0.7888, 0.7994]
2026-09-23 23:05:41.304 | INFO     | __main__

In [ ]:
!python generalization_stagewise_ct.py \
    --model resnet18 \
    --pretrained_ds imagenet \
    --transfer_ds beans \
    --seed 44 \
    --init_beta 0.8 \
    --beta_lr 0.01

2026-09-23 23:08:40.373 | INFO     | __main__:main:245 - Log file: ./logs/generalization_stagewise_ct_imagenet_to_beans_resnet18_seed44_initbeta0.8_betalr0.01.log
2026-09-23 23:08:40.433 | INFO     | __main__:main:261 - Running on cuda
Stage-Wise CT ReLU counts: [3, 2, 2, 2]
2026-09-23 23:08:42.445 | INFO     | __main__:main:312 - Initial stage betas: [0.800000011920929, 0.800000011920929, 0.800000011920929, 0.800000011920929]
2026-09-23 23:08:42.446 | INFO     | __main__:main:328 - Trainable curvature parameters: 4
2026-09-23 23:08:42.446 | INFO     | __main__:main:333 - Total trainable parameters: 1543
2026-09-23 23:08:43.734 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 1.206441, Accuracy: 34.38%
2026-09-23 23:08:51.683 | INFO     | train:test_epoch:90 - Epoch 1, Val Loss: 0.745364, Val Accuracy: 72.18%
2026-09-23 23:08:51.684 | INFO     | __main__:transfer:127 - Epoch 1: val_acc=72.18, betas=[0.8054, 0.8091, 0.7845, 0.7764]
2026-09-23 23:08:51.710 | INFO     | __main__

In [ ]:
import json
import numpy as np

seeds = [42, 43, 44]
accs = []
val_accs = []
betas = []

for seed in seeds:
    path = (
        f"results/stage_ct_imagenet_to_beans_"
        f"resnet18_seed{seed}_initbeta0.8_betalr0.01.json"
    )

    with open(path) as f:
        r = json.load(f)

    accs.append(r["accuracy"])
    val_accs.append(r["best_val_acc"])
    betas.append(r["stage_betas"])

print("Test accuracies:", accs)
print(f"Mean test accuracy: {np.mean(accs):.2f}%")
print(f"Std: {np.std(accs):.2f}")

print("\nBest validation accuracies:", val_accs)
print(f"Mean best validation accuracy: {np.mean(val_accs):.2f}%")

print("\nStage betas:")
for seed, b in zip(seeds, betas):
    print(seed, [round(x, 4) for x in b])

print(
    "\nMean stage betas:",
    np.round(np.mean(np.array(betas), axis=0), 4)
)

Test accuracies: [90.625, 89.84375, 90.625]
Mean test accuracy: 90.36%
Std: 0.37

Best validation accuracies: [96.99248120300751, 96.2406015037594, 96.99248120300751]
Mean best validation accuracy: 96.74%

Stage betas:
42 [0.8186, 0.8834, 0.8391, 0.6617]
43 [0.8152, 0.8941, 0.8102, 0.654]
44 [0.8229, 0.8846, 0.8173, 0.6435]

Mean stage betas: [0.8189 0.8874 0.8222 0.6531]


## β Initialization Sensitivity: 0.78

After selecting a β learning rate of 0.01, we now test the effect of initialization.

The original S-CT reproduction selected β values of 0.78, 0.77, and 0.79 across the three Beans seeds, giving a mean of approximately **0.78**.

Therefore, instead of initializing all four stage-wise parameters at 0.80, we initialize them at **β = 0.78** while keeping the β learning rate fixed at 0.01.

Seed 42 is evaluated first.

In [ ]:
!python generalization_stagewise_ct.py \
    --model resnet18 \
    --pretrained_ds imagenet \
    --transfer_ds beans \
    --seed 42 \
    --init_beta 0.78 \
    --beta_lr 0.01

2026-09-23 23:12:59.586 | INFO     | __main__:main:245 - Log file: ./logs/generalization_stagewise_ct_imagenet_to_beans_resnet18_seed42_initbeta0.78_betalr0.01.log
2026-09-23 23:12:59.647 | INFO     | __main__:main:261 - Running on cuda
Stage-Wise CT ReLU counts: [3, 2, 2, 2]
2026-09-23 23:13:01.927 | INFO     | __main__:main:312 - Initial stage betas: [0.7799999713897705, 0.7799999713897705, 0.7799999713897705, 0.7799999713897705]
2026-09-23 23:13:01.928 | INFO     | __main__:main:328 - Trainable curvature parameters: 4
2026-09-23 23:13:01.928 | INFO     | __main__:main:333 - Total trainable parameters: 1543
2026-09-23 23:13:03.556 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 1.032128, Accuracy: 46.88%
2026-09-23 23:13:10.672 | INFO     | train:test_epoch:90 - Epoch 1, Val Loss: 0.710428, Val Accuracy: 72.18%
2026-09-23 23:13:10.673 | INFO     | __main__:transfer:127 - Epoch 1: val_acc=72.18, betas=[0.7931, 0.7839, 0.7803, 0.7809]
2026-09-23 23:13:10.689 | INFO     | __m

### Final 4-Parameter SW-CT Configuration

The β = 0.78 initialization produced a promising result for seed 42, so this configuration is evaluated using seeds 43 and 44.

The configuration used from this point onward is:

- Initial β: **0.78**
- β learning rate: **0.01**
- Number of trainable curvature parameters: **4**
- CTU coefficient \(c\): **fixed at 0.5**

This configuration will be treated as the main 4-parameter SW-CT method for subsequent cross-dataset experiments.

In [ ]:
!python generalization_stagewise_ct.py \
    --model resnet18 \
    --pretrained_ds imagenet \
    --transfer_ds beans \
    --seed 43 \
    --init_beta 0.78 \
    --beta_lr 0.01

2026-09-23 23:16:08.731 | INFO     | __main__:main:245 - Log file: ./logs/generalization_stagewise_ct_imagenet_to_beans_resnet18_seed43_initbeta0.78_betalr0.01.log
2026-09-23 23:16:08.794 | INFO     | __main__:main:261 - Running on cuda
Stage-Wise CT ReLU counts: [3, 2, 2, 2]
2026-09-23 23:16:10.790 | INFO     | __main__:main:312 - Initial stage betas: [0.7799999713897705, 0.7799999713897705, 0.7799999713897705, 0.7799999713897705]
2026-09-23 23:16:10.791 | INFO     | __main__:main:328 - Trainable curvature parameters: 4
2026-09-23 23:16:10.791 | INFO     | __main__:main:333 - Total trainable parameters: 1543
2026-09-23 23:16:12.210 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 1.324863, Accuracy: 28.12%
2026-09-23 23:16:20.524 | INFO     | train:test_epoch:90 - Epoch 1, Val Loss: 0.762931, Val Accuracy: 70.68%
2026-09-23 23:16:20.525 | INFO     | __main__:transfer:127 - Epoch 1: val_acc=70.68, betas=[0.7859, 0.7962, 0.7649, 0.7729]
2026-09-23 23:16:20.540 | INFO     | __m

In [ ]:
!python generalization_stagewise_ct.py \
    --model resnet18 \
    --pretrained_ds imagenet \
    --transfer_ds beans \
    --seed 44 \
    --init_beta 0.78 \
    --beta_lr 0.01

2026-09-23 23:19:19.237 | INFO     | __main__:main:245 - Log file: ./logs/generalization_stagewise_ct_imagenet_to_beans_resnet18_seed44_initbeta0.78_betalr0.01.log
2026-09-23 23:19:19.303 | INFO     | __main__:main:261 - Running on cuda
Stage-Wise CT ReLU counts: [3, 2, 2, 2]
2026-09-23 23:19:21.417 | INFO     | __main__:main:312 - Initial stage betas: [0.7799999713897705, 0.7799999713897705, 0.7799999713897705, 0.7799999713897705]
2026-09-23 23:19:21.417 | INFO     | __main__:main:328 - Trainable curvature parameters: 4
2026-09-23 23:19:21.418 | INFO     | __main__:main:333 - Total trainable parameters: 1543
2026-09-23 23:19:22.800 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 1.201310, Accuracy: 31.25%
2026-09-23 23:19:31.108 | INFO     | train:test_epoch:90 - Epoch 1, Val Loss: 0.729008, Val Accuracy: 72.93%
2026-09-23 23:19:31.109 | INFO     | __main__:transfer:127 - Epoch 1: val_acc=72.93, betas=[0.7876, 0.7895, 0.7632, 0.7546]
2026-09-23 23:19:31.123 | INFO     | __m

In [ ]:
import json
import numpy as np

seeds = [42, 43, 44]

accs = []
val_accs = []
betas = []

for seed in seeds:
    path = (
        f"results/stage_ct_imagenet_to_beans_"
        f"resnet18_seed{seed}_"
        f"initbeta0.78_"
        f"betalr0.01.json"
    )

    with open(path) as f:
        r = json.load(f)

    accs.append(r["accuracy"])
    val_accs.append(r["best_val_acc"])
    betas.append(r["stage_betas"])

print("Test accuracies:", accs)
print(f"Mean test accuracy: {np.mean(accs):.2f}%")
print(f"Std: {np.std(accs):.2f}")

print("\nBest validation accuracies:", val_accs)
print(f"Mean best validation accuracy: {np.mean(val_accs):.2f}%")

print("\nStage betas:")
for seed, b in zip(seeds, betas):
    print(seed, [round(x, 4) for x in b])

print(
    "\nMean stage betas:",
    np.round(np.mean(np.array(betas), axis=0), 4)
)

Test accuracies: [90.625, 90.625, 91.40625]
Mean test accuracy: 90.89%
Std: 0.37

Best validation accuracies: [97.74436090225564, 96.2406015037594, 96.99248120300751]
Mean best validation accuracy: 96.99%

Stage betas:
42 [0.8154, 0.8787, 0.8259, 0.6574]
43 [0.8115, 0.8875, 0.8022, 0.6448]
44 [0.8152, 0.875, 0.8071, 0.6382]

Mean stage betas: [0.814  0.8804 0.8117 0.6468]


# Comparison with Higher-Parameter Adaptation Methods

After establishing the 4-parameter SW-CT configuration, additional methods are examined for reference.

## Trainable Curvature Tuning (T-CT)

T-CT allows curvature parameters to be trained at a much finer channel-wise level rather than sharing only four β values across network stages.

This provides substantially more curvature flexibility than SW-CT, but also introduces many more trainable curvature parameters.

The purpose of this experiment is to compare the parameter efficiency of the proposed 4-parameter stage-wise approach with the more flexible trainable CT formulation.

A seed-42 run is used here as an initial reference comparison.

In [ ]:
!WANDB_MODE=disabled python generalization_trainable_ct.py \
    --model resnet18 \
    --pretrained_ds imagenet \
    --transfer_ds beans \
    --seed 42

2026-09-23 23:23:28.116 | INFO     | __main__:main:94 - Log file: ./logs/generalization_trainable_ct_imagenet_to_beans_resnet18_seed42.log
2026-09-23 23:23:28.119 | INFO     | __main__:main:98 - Running on cuda
2026-09-23 23:23:30.155 | INFO     | __main__:main:124 - Testing Trainable CT...
2026-09-23 23:23:30.559 | INFO     | __main__:main:129 - Number of trainable parameters: 5507
2026-09-23 23:23:30.792 | INFO     | __main__:main:131 - Mean Beta: 0.799953, Mean Coeff: 0.500000
2026-09-23 23:23:30.793 | INFO     | __main__:main:132 - Starting transfer learning...
/usr/local/lib/python3.11/dist-packages/torch/_compile.py:32: UserWarning: optimizer contains a parameter group with duplicate parameters; in future, this will cause an error; see github.com/pytorch/pytorch/issues/40967 for more information
  return disable_fn(*args, **kwargs)
2026-09-23 23:23:32.018 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 1.194557, Accuracy: 21.88%
2026-09-23 23:23:40.408 | INFO     | tra

## LoRA Reference

LoRA is also included as a parameter-efficient adaptation reference. It modifies the network through low-rank trainable updates rather than through activation curvature.

This result is included to provide broader context for the accuracy/parameter-efficiency tradeoff. It is not the primary baseline used to define the proposed SW-CT method.

In [ ]:
!cat results/train_ct_imagenet_to_beans_resnet18_seed42.json

{
  "num_params": 5507,
  "accuracy": 90.625,
  "beta": 0.7232918739318848,
  "coeff": 0.6046077013015747
}

In [ ]:
!cat results/lora_rank1_imagenet_to_beans_resnet18_seed42.json

{
  "num_params": 37462,
  "accuracy": 92.1875
}

The 8-parameter version is implemented as a structural ablation of the main 4-parameter method. It tests whether making the coefficient \(c\) trainable provides useful additional flexibility beyond stage-wise β tuning.

The same seeds and general training procedure are retained so that the 4-parameter and 8-parameter variants can be compared directly.

In [ ]:
!WANDB_MODE=disabled python generalization_stagewise_full_ct.py \
    --model resnet18 \
    --pretrained_ds imagenet \
    --transfer_ds beans \
    --seed 42 \
    --init_beta 0.78 \
    --init_coeff 0.5 \
    --ct_lr 0.01

2026-09-23 23:29:58.804 | INFO     | __main__:main:269 - Log file: ./logs/stage_full_ct_imagenet_to_beans_resnet18_seed42_initbeta0.78_initc0.5_ctlr0.01.log
2026-09-23 23:29:58.865 | INFO     | __main__:main:282 - Running on cuda
Stage-Wise Full CT ReLU counts: [3, 2, 2, 2]
2026-09-23 23:30:00.895 | INFO     | __main__:main:338 - Initial stage betas: [0.7799999713897705, 0.7799999713897705, 0.7799999713897705, 0.7799999713897705]
2026-09-23 23:30:00.896 | INFO     | __main__:main:343 - Initial stage coeffs: [0.5, 0.5, 0.5, 0.5]
2026-09-23 23:30:00.896 | INFO     | __main__:main:371 - Trainable beta parameters: 4
2026-09-23 23:30:00.896 | INFO     | __main__:main:376 - Trainable coeff parameters: 4
2026-09-23 23:30:00.897 | INFO     | __main__:main:381 - Total trainable curvature parameters: 8
2026-09-23 23:30:00.897 | INFO     | __main__:main:386 - Total trainable parameters: 1547
2026-09-23 23:30:02.170 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 1.032128, Accuracy: 46.

In [ ]:
!WANDB_MODE=disabled python generalization_stagewise_full_ct.py \
    --model resnet18 \
    --pretrained_ds imagenet \
    --transfer_ds beans \
    --seed 43 \
    --init_beta 0.78 \
    --init_coeff 0.5 \
    --ct_lr 0.01

2026-09-23 23:33:10.526 | INFO     | __main__:main:269 - Log file: ./logs/stage_full_ct_imagenet_to_beans_resnet18_seed43_initbeta0.78_initc0.5_ctlr0.01.log
2026-09-23 23:33:10.585 | INFO     | __main__:main:282 - Running on cuda
Stage-Wise Full CT ReLU counts: [3, 2, 2, 2]
2026-09-23 23:33:12.575 | INFO     | __main__:main:338 - Initial stage betas: [0.7799999713897705, 0.7799999713897705, 0.7799999713897705, 0.7799999713897705]
2026-09-23 23:33:12.575 | INFO     | __main__:main:343 - Initial stage coeffs: [0.5, 0.5, 0.5, 0.5]
2026-09-23 23:33:12.575 | INFO     | __main__:main:371 - Trainable beta parameters: 4
2026-09-23 23:33:12.576 | INFO     | __main__:main:376 - Trainable coeff parameters: 4
2026-09-23 23:33:12.576 | INFO     | __main__:main:381 - Total trainable curvature parameters: 8
2026-09-23 23:33:12.576 | INFO     | __main__:main:386 - Total trainable parameters: 1547
2026-09-23 23:33:13.991 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 1.324863, Accuracy: 28.

In [ ]:
!WANDB_MODE=disabled python generalization_stagewise_full_ct.py \
    --model resnet18 \
    --pretrained_ds imagenet \
    --transfer_ds beans \
    --seed 44 \
    --init_beta 0.78 \
    --init_coeff 0.5 \
    --ct_lr 0.01

2026-09-23 23:36:23.725 | INFO     | __main__:main:269 - Log file: ./logs/stage_full_ct_imagenet_to_beans_resnet18_seed44_initbeta0.78_initc0.5_ctlr0.01.log
2026-09-23 23:36:23.790 | INFO     | __main__:main:282 - Running on cuda
Stage-Wise Full CT ReLU counts: [3, 2, 2, 2]
2026-09-23 23:36:26.091 | INFO     | __main__:main:338 - Initial stage betas: [0.7799999713897705, 0.7799999713897705, 0.7799999713897705, 0.7799999713897705]
2026-09-23 23:36:26.091 | INFO     | __main__:main:343 - Initial stage coeffs: [0.5, 0.5, 0.5, 0.5]
2026-09-23 23:36:26.092 | INFO     | __main__:main:371 - Trainable beta parameters: 4
2026-09-23 23:36:26.092 | INFO     | __main__:main:376 - Trainable coeff parameters: 4
2026-09-23 23:36:26.092 | INFO     | __main__:main:381 - Total trainable curvature parameters: 8
2026-09-23 23:36:26.092 | INFO     | __main__:main:386 - Total trainable parameters: 1547
2026-09-23 23:36:27.524 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 1.201310, Accuracy: 31.

In [ ]:
import json
import numpy as np

seeds = [42, 43, 44]

accs = []
val_accs = []
betas = []
coeffs = []

for seed in seeds:
    path = (
        f"results/stage_full_ct_imagenet_to_beans_"
        f"resnet18_seed{seed}_"
        f"initbeta0.78_initc0.5_ctlr0.01.json"
    )

    with open(path) as f:
        r = json.load(f)

    accs.append(r["accuracy"])
    val_accs.append(r["best_val_acc"])
    betas.append(r["stage_betas"])
    coeffs.append(r["stage_coeffs"])

print("Test accuracies:", accs)
print(f"Mean test accuracy: {np.mean(accs):.2f}%")
print(f"Std: {np.std(accs):.2f}")

print("\nBest validation accuracies:", val_accs)
print(f"Mean best validation accuracy: {np.mean(val_accs):.2f}%")

print("\nStage betas:")
for seed, b in zip(seeds, betas):
    print(seed, [round(x, 4) for x in b])

print(
    "Mean stage betas:",
    np.round(np.mean(betas, axis=0), 4)
)

print("\nStage coeffs:")
for seed, c in zip(seeds, coeffs):
    print(seed, [round(x, 4) for x in c])

print(
    "Mean stage coeffs:",
    np.round(np.mean(coeffs, axis=0), 4)
)

Test accuracies: [89.84375, 91.40625, 91.40625]
Mean test accuracy: 90.89%
Std: 0.74

Best validation accuracies: [96.99248120300751, 96.2406015037594, 96.99248120300751]
Mean best validation accuracy: 96.74%

Stage betas:
42 [0.8007, 0.8675, 0.8347, 0.6568]
43 [0.7935, 0.8819, 0.8173, 0.6379]
44 [0.8006, 0.864, 0.8179, 0.6411]
Mean stage betas: [0.7983 0.8711 0.8233 0.6453]

Stage coeffs:
42 [0.6678, 0.6609, 0.3381, 0.523]
43 [0.6965, 0.6046, 0.3573, 0.5606]
44 [0.6613, 0.675, 0.3609, 0.5011]
Mean stage coeffs: [0.6752 0.6468 0.3521 0.5282]


### 8-Parameter Ablation Result

The 8-parameter variant did not improve mean Beans test accuracy over the 4-parameter SW-CT configuration.

Both variants achieved approximately **90.89% mean test accuracy**, while the 8-parameter version showed greater variation across seeds.

Therefore, the additional trainable stage-wise \(c\) parameters did not provide a clear accuracy benefit on Beans. The simpler 4-parameter β-only formulation remains the main proposed method, while the 8-parameter version is retained as an ablation.